# 00 - Plano de Coordenação e Orquestração Multi-Agente

## 1. Cabeçalho
- **Agente:** Agente Coordenador (Sistema Multi-Agente Antigravity)
- **Fase CRISP-DM:** Visão Global, Governança & Arquitetura (Todas as 6 Fases)
- **Data e Hora de Criação:** 2026-10-23 21:30 UTC
- **Idioma:** Português Europeu (relatórios e documentação) / Inglês (código e nomes de variáveis)
- **Ambiente:** Antigravity / Python Local (100% dos Agentes 1 a 7 e Coordenador)
- **Versões das Bibliotecas:**
  - Python: 3.14.0
  - Pandas: 3.0.6
  - Polars: 2.0.0
  - PyArrow: 26.0.0
  - PyYAML: 6.0.3
  - Requests: 2.34.2


## 2. Objetivo e Critérios de Sucesso

### 2.1 Objetivo Geral
Orquestrar de ponta a ponta a construção de um projeto de Data Science de qualidade de portfólio para **explicar e prever o preço horário da eletricidade no mercado diário (MIBEL, zona de Portugal)**, integrando dados reais de APIs públicas (ENTSO-E, Open-Meteo, OMIE, mercados de combustíveis) e aplicando modelos XGBoost com interpretabilidade global e local via SHAP.

### 2.2 Critérios de Sucesso do Coordenador
1. **Governança CRISP-DM:** Cumprimento rigoroso das 6 fases do ciclo de vida, sem quebras metodológicas.
2. **Reprodutibilidade Total:** Configuração centralizada em `config.yaml`, seeds fixadas, dependências declaradas e pipeline executável do início ao fim (*Restart & Run All* sem erros).
3. **Quality Gates Invioláveis:** Nenhum agente inicia o seu trabalho sem que o Quality Gate do agente precedente esteja formalmente validado e registado em `logs/gate_results.json`.
4. **Prevenção de Data Leakage:** Garantia de que nenhuma feature utilize dados com timestamp de disponibilização posterior ao momento de fecho do mercado diário ($D-1$ ~12:00 CET).
5. **Transparência e Rastreabilidade:** Registo de qualquer transformação de dados em `logs/correction_log.csv` e diário de operações em `logs/run_log.md`.
6. **Gestão de Checkpoints Humanos:** Paragens obrigatórias para alinhamento com o utilizador nos 3 marcos críticos definidos.


## 3. Inputs

| Ficheiro / Recurso | Tipo | Origem | Finalidade |
|---|---|---|---|
| `config.yaml` | Ficheiro de Configuração | Desenvolvido pelo Coordenador | Parametrização global (datas, fusos, caminhos, fontes, limiares dos gates) |
| `.env.example` | Template de Ambiente | Desenvolvido pelo Coordenador | Especificação de segredos e chaves de API necessárias |
| Especificação de Execução Local | Web / Documentação Oficial | Ambiente Antigravity Python | Avaliação de limites técnicos (PAT, Jobs API, Volumes, MLflow) |
| Especificação do Prompt Mestre | Requisitos do Projeto | Utilizador | Matriz de papéis dos 7 agentes e regras operacionais |


## 4. Decisões e Justificação

| Domínio da Decisão | Decisão Tomada | Alternativas Consideradas | Justificação Técnica e de Negócio |
|---|---|---|---|
| **Estratégia de Execução** | **Execução 100% Local (Antigravity)** | Todo o pipeline de ingestão, features, 100 treinos XGBoost e SHAP corre localmente sem dependência de cloud. |
| **Formato de Dados** | **Apache Parquet (compressão Snappy)** | CSV comprimido, SQLite, Delta Lake local | Parquet preserva schemas de dados (tipos estritos, precisão decimal, metadata temporal), oferece elevada taxa de compressão e é o padrão nativo de ingestão no Ambiente Local. |
| **Referência Temporal** | Duas colunas obrigatórias: `timestamp_utc` (UTC canónico) e `timestamp_local` (`Europe/Lisbon`) | Apenas UTC ou apenas hora local | Evita perdas ou duplicações nas mudanças sazonais de hora (DST - *Daylight Saving Time*). O UTC garante ordenação contínua estrita; o fuso local preserva o alinhamento com os hábitos de consumo e hábitos solares em Portugal. |
| **Enquadramento de ML** | **Regressão Contínua do Preço Horário (€/MWh)** (com validação temporal walk-forward) | Classificação Binária (Preço Alto vs Normal) ou Classificação de Preços Negativos | A regressão contínua fornece a máxima utilidade financeira e analítica (curva de custos de produção, receitas de centrais renováveis e comercializadores). Os regimes de preços negativos e picos são analisados como casos específicos de resíduos e SHAP. |
| **Resolução Temporal** | **Horária (1h)** com agregação aritmética para dados posteriores a 01/10/2025 | Granularidade de 15 minutos em todo o dataset | O mercado europeu (SDAC/MIBEL) migrou para 15 minutos em 01/10/2025 ("Big Bang"). Para manter comparabilidade uniforme com o histórico anterior de 2023-2025, o passo horário é o denominador comum ideal, calculando a média aritmética dos 4 períodos de 15 min. |


In [1]:
# 5. Execução: Validação do Ambiente e Estrutura do Projeto
import sys
import os
import json
import yaml
import pandas as pd
from pathlib import Path

# Adicionar pasta src ao sys.path
project_root = Path.cwd().parent if Path.cwd().name == "reports" else Path.cwd()
sys.path.append(str(project_root))

from src.utils import load_config, get_project_root, calculate_file_hash

print(f"[OK] Python Version: {sys.version}")
print(f"[OK] Project Root: {project_root}")

# Carregar e validar configuração central
config = load_config()
print(f"[OK] Projeto: {config['project']['name']} (Versão {config['project']['version']})")
print(f"[OK] Modo de Execução Selecionado: {config.get('execution_environment', {}).get('mode', 'local')}")
print(f"[OK] Janela Temporal: {config['temporal_settings']['dataset_start']} até {config['temporal_settings']['dataset_end']}")
print(f"[OK] Enquadramento ML: {config['ml_framing']['approved_framing']}")


[OK] Python Version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
[OK] Project Root: 
[OK] Projeto: energy-price-agents-mibel-pt (Versão 1.0.0)
[OK] Plano Ambiente Local Selecionado: Plano B
[OK] Janela Temporal: 2023-01-01 até 2026-09-30
[OK] Enquadramento ML: regression_hourly_price


In [2]:
# Verificação das pastas do projeto
expected_dirs = [
    "data/raw", "data/interim", "data/processed",
    "reports", "logs", "src"
]

missing_dirs = []
for d in expected_dirs:
    dir_path = project_root / d
    if dir_path.exists() and dir_path.is_dir():
        print(f"  [Diretório Existente] {d}")
    else:
        missing_dirs.append(d)

if not missing_dirs:
    print("\n[SUCESSO] Todos os diretórios obrigatórios da arquitetura estão criados!")
else:
    print(f"\n[ERRO] Diretórios em falta: {missing_dirs}")


  [Diretório Existente] data/raw
  [Diretório Existente] data/interim
  [Diretório Existente] data/processed
  [Diretório Existente] reports
  [Diretório Existente] logs
  [Diretório Existente] src

[SUCESSO] Todos os diretórios obrigatórios da arquitetura estão criados!


In [3]:
# Validação dos ficheiros de logs e contratos de qualidade
logs_files = ["logs/gate_results.json", "logs/run_log.md", "logs/correction_log.csv"]
for lf in logs_files:
    file_path = project_root / lf
    if file_path.exists():
        size_bytes = file_path.stat().st_size
        print(f"  [Log Ativo] {lf} ({size_bytes} bytes)")
    else:
        print(f"  [FALHA] Ficheiro não encontrado: {lf}")


  [Log Ativo] logs/gate_results.json (1495 bytes)
  [Log Ativo] logs/run_log.md (1562 bytes)
  [Log Ativo] logs/correction_log.csv (100 bytes)


In [4]:
# Visualização do estado atual dos Quality Gates
with open(project_root / "logs" / "gate_results.json", "r", encoding="utf-8") as f:
    gates_state = json.load(f)

print("Estado Atual dos Quality Gates:")
for gate_id, gate_info in gates_state["gates"].items():
    print(f"  - {gate_id} ({gate_info['from_agent']} -> {gate_info['to_agent']}): Status = {gate_info['status']}")


Estado Atual dos Quality Gates:
  - Gate_1 (Agente 1 -> Agente 2): Status = IN_REVIEW
  - Gate_2 (Agente 2 -> Agente 3): Status = NOT_STARTED
  - Gate_3 (Agente 3 -> Agente 4/5): Status = NOT_STARTED
  - Gate_5 (Agente 5 -> Agente 6): Status = NOT_STARTED
  - Gate_6 (Agente 6 -> Agente 7): Status = NOT_STARTED


## 6. Problemas, Riscos e Limitações Encontrados

1. **Execução Exclusivamente Local: Todo o processamento e treino corre de forma autónoma e reprodutível na máquina local.
   - *Risco:* Ausência de PATs inviabiliza orquestração headless 100% remota via CLI.
   - *Mitigação:* Execução 100% local no Antigravity, eliminando credenciais de cloud e garantindo autonomia total.

2. **Transição de Granularidade de Mercado (SDAC 15 min):**
   - *Risco:* Mistura de registos com 24 observações diárias (até set/2025) e 96 observações diárias (a partir de out/2025).
   - *Mitigação:* Definição explícita no `config.yaml` da regra de agregação horária por média aritmética dos 4 blocos de 15 min, garantindo um índice temporal contínuo sem quebras de dimensionalidade.

3. **Chaves de Acesso a APIs Externas:**
   - *Risco:* Algumas APIs oficiais (como ENTSO-E) exigem registo e aprovação de chave de API por email.
   - *Mitigação:* O Agente 1 mapeou fontes alternativas diretas (como dados públicos do OMIE e Open-Meteo, que operam sem chave) como salvaguarda e redundância.


In [5]:
# 7. Outputs Gerados e Checksums
outputs = [
    "config.yaml",
    ".env.example",
    "logs/gate_results.json",
    "logs/run_log.md",
    "logs/correction_log.csv",
    "src/utils.py"
]

print("Tabela de Outputs e Hashes de Integridade (SHA-256):\n")
summary_data = []
for out in outputs:
    full_path = project_root / out
    if full_path.exists():
        h = calculate_file_hash(full_path)
        size = full_path.stat().st_size
        summary_data.append({"Ficheiro": out, "Tamanho (Bytes)": size, "SHA-256": h[:16] + "..."})

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_markdown(index=False))


Tabela de Outputs e Hashes de Integridade (SHA-256):



| Ficheiro                |   Tamanho (Bytes) | SHA-256             |
|:------------------------|------------------:|:--------------------|
| config.yaml             |              4628 | 844a1b64a06bff27... |
| .env.example            |               592 | f188bb203ffb9c79... |
| logs/gate_results.json  |              1495 | 678477e0353d909b... |
| logs/run_log.md         |              1562 | 4065765d94776bf8... |
| logs/correction_log.csv |               100 | ecc70bd91d34d108... |
| src/utils.py            |              4072 | 4d5593a37ee685ba... |


## 8. Resultado do Quality Gate do Agente

| Verificação do Gate 0 (Setup & Governança) | Critério de Aceitação | Estado | Evidência |
|---|---|---|---|
| Estrutura de Diretórios | Todas as 6 pastas criadas | **PASS** | Pastas `data/raw`, `interim`, `processed`, `reports`, `logs`, `src` verificadas |
| Ficheiro `config.yaml` | Completo, validado e conforme o prompt | **PASS** | Parametrização temporal, enquadramento ML e regras de gates definidas |
| Template `.env.example` | Criado sem dados sensíveis expostos | **PASS** | Ficheiro presente com placeholders para ENTSO-E, Ambiente Local e REN |
| Modo de Execução | Execução 100% Local formalizada | **PASS** | Registado no `config.yaml` |
| Ficheiros de Log | Criados com headers de contrato | **PASS** | `run_log.md`, `gate_results.json`, `correction_log.csv` inicializados |

**Decisão do Gate 0:** **APROVADO (PASS)**.


## 9. Recomendações para o Agente Seguinte (Agente 1)

1. **Execução Local: Garantir que todo o pipeline e treino de 100 modelos corre localmente sem credenciais externas.
2. **Avaliação Ponderada de Fontes:** Aplicar a fórmula de pontuação (Score 1 a 5, corte em 3,5) às seguintes fontes prioritárias:
   - ENTSO-E Transparency Platform (Preços, Geração por tipo, Carga)
   - Open-Meteo API (Radiação Solar, Vento a 100m, Temperatura e Cobertura de Nuvens)
   - OMIE (Preços marginais horários directos de Portugal)
   - REN Data Hub (Balanço do sistema electroprodutor de Portugal)
   - Commodities (Preços de gás TTF e Licenças de emissão de CO2)
3. **Resolução de 15 Minutos:** Documentar em pormenor a transição de 01/10/2025 no mercado diário MIBEL.
4. **Proposta de Enquadramento ML:** Formalizar a comparação entre a regressão contínua do preço e as abordagens de classificação/anomalias.
5. **Preparação para o Checkpoint Humano (a):** Reunir num formato executivo claro as conclusões para apresentação e validação pelo utilizador.
